# BST & SASRec: Transformer-based Sequential Models on Taobao

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)
[![BST Paper](https://img.shields.io/badge/Paper-BST%20(DLP--KDD%202019)-green)](https://arxiv.org/abs/1905.06874)
[![SASRec Paper](https://img.shields.io/badge/Paper-SASRec%20(ICDM%202018)-blue)](https://arxiv.org/abs/1808.09781)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Compare** DIN's target-aware attention with Transformer self-attention
2. **Implement** BST (Behavior Sequence Transformer) for CTR prediction
3. **Implement** SASRec (Self-Attentive Sequential Recommendation) for next-item prediction
4. **Understand** the role of positional encoding and causal masking
5. **Visualize** multi-head attention patterns and compare with DIN

## Prerequisites

- Completed Notebooks 01 (data) and 02 (DIN)
- Understanding of Transformer architecture fundamentals
- PyTorch, GPU recommended

## 1. Theory: Transformers for Sequential Recommendation

### DIN vs Transformer Self-Attention

| Aspect | DIN | Transformer (BST/SASRec) |
|--------|-----|-------------------------|
| Attention type | Target-to-history (unidirectional) | History-to-history (bidirectional or causal) |
| What it captures | "Which past items relate to the target?" | "How do past items relate to *each other*?" |
| Complexity | $O(H)$ per target | $O(H^2)$ for full self-attention |
| Positional info | Not explicit | Positional embeddings |

### BST: Behavior Sequence Transformer

> **Concept:** BST applies a Transformer encoder over the user's behavior sequence, then concatenates the Transformer output with the target item embedding for CTR prediction. The key insight is that self-attention captures *pairwise item relationships* within the sequence.

The BST architecture:

$$\mathbf{H} = \text{TransformerEncoder}([e_1 + p_1, e_2 + p_2, \ldots, e_H + p_H])$$

$$\hat{y} = \text{MLP}([\text{mean}(\mathbf{H}), e_t])$$

where $p_i$ are learnable positional embeddings.

### SASRec: Self-Attentive Sequential Recommendation

> **Concept:** SASRec uses **causal (left-to-right) self-attention** to predict the next item at each position in the sequence. Unlike BST which uses bidirectional attention for CTR, SASRec applies a **causal mask** to prevent information leakage from future items.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$

where $M$ is the causal mask:

$$M_{ij} = \begin{cases} 0 & \text{if } i \geq j \\ -\infty & \text{if } i < j \end{cases}$$

### Positional Encoding

Unlike RNNs, Transformers have no inherent notion of order. We add **learnable positional embeddings** to encode the position of each item in the sequence:

$$\tilde{e}_i = e_i + p_i$$

where $p_i \in \mathbb{R}^d$ is a learnable vector for position $i$.

## 2. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import time
import math
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, log_loss

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: NVIDIA GeForce RTX 3060


## 3. Load Data

In [2]:
PROCESSED_DIR = '../../data/taobao/processed/'
with open(os.path.join(PROCESSED_DIR, 'taobao_sequential_data.pkl'), 'rb') as f:
    data = pickle.load(f)

user_sequences = data['user_sequences']
n_items = data['n_items']
n_categories = data['n_categories']
item_popularity = data['item_popularity']

# Build item-to-category mapping
item_to_cat = {}
for uid, seq in user_sequences.items():
    for item_id, cat_id in zip(seq['item_ids'], seq['cat_ids']):
        item_to_cat[item_id] = cat_id

all_item_ids = list(item_popularity.keys())
all_item_probs = np.array([item_popularity[i] for i in all_item_ids])
all_item_probs = all_item_probs / all_item_probs.sum()

MAX_SEQ_LEN = 50
NEG_RATIO = 4

print(f'Loaded: {len(user_sequences)} users, {n_items} items, {n_categories} categories')

Loaded: 93297 users, 410453 items, 5200 categories


In [3]:
# Reuse CTR dataset from DIN notebook
class TaobaoCTRDataset(Dataset):
    """Dataset for CTR prediction with positive/negative sampling."""
    
    def __init__(self, user_sequences, item_to_cat, all_item_ids, all_item_probs,
                 max_seq_len=50, neg_ratio=4, mode='train'):
        self.max_seq_len = max_seq_len
        self.neg_ratio = neg_ratio
        self.item_to_cat = item_to_cat
        self.all_item_ids = np.array(all_item_ids)
        self.all_item_probs = all_item_probs
        self.mode = mode
        self.samples = []
        self._build_samples(user_sequences)
    
    def _build_samples(self, user_sequences):
        for uid, seq in user_sequences.items():
            items = seq['item_ids']
            cats = seq['cat_ids']
            if len(items) < 3:
                continue
            
            target_idx = len(items) - 2 if self.mode == 'train' else len(items) - 1
            hist_items = items[:target_idx]
            hist_cats = cats[:target_idx]
            if len(hist_items) == 0:
                continue
            
            if len(hist_items) > self.max_seq_len:
                hist_items = hist_items[-self.max_seq_len:]
                hist_cats = hist_cats[-self.max_seq_len:]
            
            hist_len = len(hist_items)
            pad_len = self.max_seq_len - hist_len
            hist_items_padded = [0] * pad_len + hist_items
            hist_cats_padded = [0] * pad_len + hist_cats
            
            target_item = items[target_idx]
            target_cat = cats[target_idx]
            self.samples.append((hist_items_padded, hist_cats_padded, hist_len,
                                target_item, target_cat, 1))
            
            user_item_set = set(items)
            neg_count = 0
            max_tries = self.neg_ratio * 10
            tries = 0
            while neg_count < self.neg_ratio and tries < max_tries:
                neg_item = np.random.choice(self.all_item_ids, p=self.all_item_probs)
                tries += 1
                if neg_item not in user_item_set:
                    neg_cat = self.item_to_cat.get(neg_item, 0)
                    self.samples.append((hist_items_padded, hist_cats_padded, hist_len,
                                        neg_item, neg_cat, 0))
                    neg_count += 1
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        hist_items, hist_cats, hist_len, target_item, target_cat, label = self.samples[idx]
        return {
            'hist_items': torch.LongTensor(hist_items),
            'hist_cats': torch.LongTensor(hist_cats),
            'hist_len': torch.LongTensor([hist_len]),
            'target_item': torch.LongTensor([target_item]),
            'target_cat': torch.LongTensor([target_cat]),
            'label': torch.FloatTensor([label])
        }

print('Building CTR datasets...')
train_dataset = TaobaoCTRDataset(
    user_sequences, item_to_cat, all_item_ids, all_item_probs,
    max_seq_len=MAX_SEQ_LEN, neg_ratio=NEG_RATIO, mode='train'
)
test_dataset = TaobaoCTRDataset(
    user_sequences, item_to_cat, all_item_ids, all_item_probs,
    max_seq_len=MAX_SEQ_LEN, neg_ratio=NEG_RATIO, mode='test'
)
print(f'Train: {len(train_dataset):,}, Test: {len(test_dataset):,}')

BATCH_SIZE = 1024
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

Building CTR datasets...


## 4. BST Implementation

> **Concept:** BST applies a standard Transformer encoder over the behavior sequence, using **bidirectional self-attention** (each item can attend to all other items). The Transformer output is then combined with the target item for CTR prediction.

> **Common Pitfall:** When using Transformer with padded sequences, you **must** pass a key_padding_mask to prevent padded positions from influencing the attention computation. Failing to mask padding is a common source of degraded performance.

In [ ]:
class BST(nn.Module):
    """Behavior Sequence Transformer for CTR Prediction.
    
    Architecture:
        Item + Category Embedding + Positional Embedding
        -> Transformer Encoder (multi-head self-attention)
        -> Mean pooling over sequence
        -> Concat with target item embedding
        -> MLP -> Click probability
    """
    
    def __init__(self, n_items, n_categories, embed_dim=32, n_heads=4, n_layers=2,
                 max_seq_len=50, hidden_dims=[256, 128, 64], dropout=0.2):
        super().__init__()
        
        self.embed_dim = embed_dim
        cat_dim = embed_dim // 2
        self.feature_dim = embed_dim + cat_dim
        
        # Embeddings
        self.item_embedding = nn.Embedding(n_items, embed_dim, padding_idx=0)
        self.cat_embedding = nn.Embedding(n_categories, cat_dim, padding_idx=0)
        self.position_embedding = nn.Embedding(max_seq_len, self.feature_dim)
        
        # Layer norm and dropout for embeddings
        self.emb_layernorm = nn.LayerNorm(self.feature_dim)
        self.emb_dropout = nn.Dropout(dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.feature_dim,
            nhead=n_heads,
            dim_feedforward=self.feature_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True  # Pre-LN Transformer (more stable training)
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers
        )
        
        # MLP for prediction
        mlp_input_dim = self.feature_dim * 2  # transformer output + target
        layers = []
        in_dim = mlp_input_dim
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(in_dim, h_dim),
                nn.LayerNorm(h_dim),
                nn.GELU(),
                nn.Dropout(dropout)
            ])
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)
                if m.padding_idx is not None:
                    nn.init.zeros_(m.weight[m.padding_idx])
    
    def forward(self, hist_items, hist_cats, hist_len, target_item, target_cat,
                return_attention=False):
        B, S = hist_items.shape
        
        # Item + Category embeddings
        item_emb = self.item_embedding(hist_items)   # (B, S, D)
        cat_emb = self.cat_embedding(hist_cats)      # (B, S, D/2)
        seq_emb = torch.cat([item_emb, cat_emb], dim=-1)  # (B, S, D+D/2)
        
        # Add positional embeddings
        positions = torch.arange(S, device=hist_items.device).unsqueeze(0).expand(B, S)
        pos_emb = self.position_embedding(positions)  # (B, S, D+D/2)
        seq_emb = seq_emb + pos_emb
        
        seq_emb = self.emb_layernorm(seq_emb)
        seq_emb = self.emb_dropout(seq_emb)
        
        # Padding mask: True for padded positions
        padding_mask = (hist_items == 0)  # (B, S)
        
        # Transformer Encoder
        transformer_out = self.transformer_encoder(
            seq_emb, src_key_padding_mask=padding_mask
        )  # (B, S, D+D/2)
        
        # Mean pooling over valid positions
        mask_float = (~padding_mask).float().unsqueeze(-1)  # (B, S, 1)
        pooled = (transformer_out * mask_float).sum(dim=1) / mask_float.sum(dim=1).clamp(min=1)
        
        # Target embedding
        target_item_emb = self.item_embedding(target_item).squeeze(1)
        target_cat_emb = self.cat_embedding(target_cat).squeeze(1)
        target_emb = torch.cat([target_item_emb, target_cat_emb], dim=-1)
        
        # Concat and predict
        mlp_input = torch.cat([pooled, target_emb], dim=-1)
        logits = self.mlp(mlp_input)
        
        return logits, None


# Instantiate BST
bst_model = BST(
    n_items=n_items,
    n_categories=n_categories,
    embed_dim=32,
    n_heads=4,
    n_layers=2,
    max_seq_len=MAX_SEQ_LEN,
    hidden_dims=[256, 128, 64],
    dropout=0.2
).to(device)

total_params = sum(p.numel() for p in bst_model.parameters())
print(f'BST parameters: {total_params:,}')
print(bst_model)

## 5. Train BST

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    n_batches = 0
    for batch in loader:
        hist_items = batch['hist_items'].to(device)
        hist_cats = batch['hist_cats'].to(device)
        hist_len = batch['hist_len'].to(device)
        target_item = batch['target_item'].to(device)
        target_cat = batch['target_cat'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        logits, _ = model(hist_items, hist_cats, hist_len, target_item, target_cat)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    n_batches = 0
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            hist_items = batch['hist_items'].to(device)
            hist_cats = batch['hist_cats'].to(device)
            hist_len = batch['hist_len'].to(device)
            target_item = batch['target_item'].to(device)
            target_cat = batch['target_cat'].to(device)
            labels = batch['label'].to(device)
            
            logits, _ = model(hist_items, hist_cats, hist_len, target_item, target_cat)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            n_batches += 1
            preds = torch.sigmoid(logits).cpu().numpy().flatten()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy().flatten())
    
    avg_loss = total_loss / n_batches
    auc = roc_auc_score(all_labels, all_preds)
    logloss = log_loss(all_labels, np.clip(all_preds, 1e-7, 1-1e-7))
    return avg_loss, auc, logloss, np.array(all_labels), np.array(all_preds)

In [ ]:
# Training configuration
MAX_EPOCHS = 15
PATIENCE = 3
LR = 1e-3
WEIGHT_DECAY = 1e-6

criterion = nn.BCEWithLogitsLoss()
bst_optimizer = torch.optim.Adam(bst_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
bst_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(bst_optimizer, mode='max', patience=1, factor=0.5)

bst_best_auc = 0
patience_counter = 0
bst_history = {'train_loss': [], 'test_loss': [], 'test_auc': [], 'test_logloss': []}

print(f'Training BST for up to {MAX_EPOCHS} epochs...')
print('='*70)

for epoch in range(MAX_EPOCHS):
    start_time = time.time()
    
    train_loss = train_epoch(bst_model, train_loader, bst_optimizer, criterion, device)
    test_loss, test_auc, test_logloss, _, _ = evaluate(bst_model, test_loader, criterion, device)
    
    epoch_time = time.time() - start_time
    bst_scheduler.step(test_auc)
    
    bst_history['train_loss'].append(train_loss)
    bst_history['test_loss'].append(test_loss)
    bst_history['test_auc'].append(test_auc)
    bst_history['test_logloss'].append(test_logloss)
    
    improved = ''
    if test_auc > bst_best_auc:
        bst_best_auc = test_auc
        patience_counter = 0
        torch.save(bst_model.state_dict(), os.path.join(PROCESSED_DIR, 'bst_best.pt'))
        improved = ' *'
    else:
        patience_counter += 1
    
    print(f'Epoch {epoch+1:2d}/{MAX_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Test Loss: {test_loss:.4f} | '
          f'Test AUC: {test_auc:.4f} | '
          f'Test LogLoss: {test_logloss:.4f} | '
          f'Time: {epoch_time:.1f}s{improved}')
    
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break

print(f'\nBST Best Test AUC: {bst_best_auc:.4f}')

In [ ]:
# Plot BST training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs = range(1, len(bst_history['train_loss']) + 1)

axes[0].plot(epochs, bst_history['train_loss'], 'o-', label='Train', color='#4e79a7')
axes[0].plot(epochs, bst_history['test_loss'], 's-', label='Test', color='#e15759')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('BST: Training & Test Loss')
axes[0].legend()

axes[1].plot(epochs, bst_history['test_auc'], 'o-', color='#59a14f', linewidth=2)
axes[1].axhline(0.64, color='red', linestyle='--', alpha=0.7, label='Target AUC=0.64')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].set_title('BST: Test AUC')
axes[1].legend()

axes[2].plot(epochs, bst_history['test_logloss'], 'o-', color='#f28e2b', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('LogLoss')
axes[2].set_title('BST: Test LogLoss')

plt.tight_layout()
plt.savefig('bst_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. SASRec Implementation

> **Concept:** SASRec differs from BST in two fundamental ways:
> 1. **Causal masking**: Each position can only attend to current and previous positions (left-to-right)
> 2. **Task**: Instead of CTR prediction, SASRec predicts the *next item* at each position using the sequence representation

> **Pro Tip:** SASRec's causal attention makes it suitable for autoregressive generation of recommendations, similar to how GPT generates text token by token.

In [ ]:
class SASRec(nn.Module):
    """Self-Attentive Sequential Recommendation.
    
    Uses causal (left-to-right) self-attention for next-item prediction.
    """
    
    def __init__(self, n_items, embed_dim=64, n_heads=2, n_layers=2,
                 max_seq_len=50, dropout=0.2):
        super().__init__()
        
        self.n_items = n_items
        self.embed_dim = embed_dim
        self.max_seq_len = max_seq_len
        
        # Item embedding (shared for input and prediction)
        self.item_embedding = nn.Embedding(n_items, embed_dim, padding_idx=0)
        self.position_embedding = nn.Embedding(max_seq_len, embed_dim)
        
        self.emb_layernorm = nn.LayerNorm(embed_dim)
        self.emb_dropout = nn.Dropout(dropout)
        
        # Transformer Encoder with causal masking
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers
        )
        
        self.output_layernorm = nn.LayerNorm(embed_dim)
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)
                if m.padding_idx is not None:
                    nn.init.zeros_(m.weight[m.padding_idx])
    
    def _generate_causal_mask(self, sz, device):
        """Generate causal mask: position i can attend to positions <= i."""
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
        return mask  # True = masked out
    
    def forward(self, input_ids, return_all_positions=False):
        """
        Args:
            input_ids: (B, S) - item ID sequence
            return_all_positions: if True, return representations for all positions
        Returns:
            output: (B, D) or (B, S, D) - sequence representations
        """
        B, S = input_ids.shape
        
        # Embeddings
        item_emb = self.item_embedding(input_ids)  # (B, S, D)
        positions = torch.arange(S, device=input_ids.device).unsqueeze(0).expand(B, S)
        pos_emb = self.position_embedding(positions)  # (B, S, D)
        
        seq_emb = item_emb + pos_emb
        seq_emb = self.emb_layernorm(seq_emb)
        seq_emb = self.emb_dropout(seq_emb)
        
        # Masks
        causal_mask = self._generate_causal_mask(S, input_ids.device)
        padding_mask = (input_ids == 0)  # (B, S)
        
        # Transformer with causal mask
        output = self.transformer_encoder(
            seq_emb,
            mask=causal_mask,
            src_key_padding_mask=padding_mask
        )  # (B, S, D)
        
        output = self.output_layernorm(output)
        
        if return_all_positions:
            return output
        else:
            # Return last valid position representation
            return output  # We'll extract the last position outside
    
    def predict(self, seq_output, target_items):
        """
        Compute scores for target items.
        
        Args:
            seq_output: (B, D) - sequence representation
            target_items: (B, K) - target item IDs (K candidates)
        Returns:
            scores: (B, K) - dot-product scores
        """
        target_emb = self.item_embedding(target_items)  # (B, K, D)
        # Dot product
        scores = (seq_output.unsqueeze(1) * target_emb).sum(dim=-1)  # (B, K)
        return scores


sasrec_model = SASRec(
    n_items=n_items,
    embed_dim=64,
    n_heads=2,
    n_layers=2,
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.2
).to(device)

total_params = sum(p.numel() for p in sasrec_model.parameters())
print(f'SASRec parameters: {total_params:,}')
print(sasrec_model)

> **Common Pitfall: Left-Padding + Causal Mask = NaN**
>
> SASRec uses a causal (lower-triangular) attention mask so each position can only
> attend to itself and earlier positions. If we left-pad (zeros at the start), the
> first few positions have **all** keys masked (padded), causing softmax to compute
> 0/0 = NaN. This NaN propagates through all transformer layers and corrupts model
> weights via gradient flow.
>
> **Solution:** Use **right-padding** (items first, zeros after). With right-padding,
> position 0 is always a valid item, so every position has at least one attendable key.
> We also use  instead of  to avoid
> IEEE 754 NaN × 0 = NaN in the loss computation.

In [ ]:
# SASRec Dataset: next-item prediction with BPR loss
class SASRecDataset(Dataset):
    """Dataset for SASRec next-item prediction.
    
    For each user, the input is the sequence [i1, i2, ..., i_{n-1}]
    and the target is [i2, i3, ..., i_n].
    We also sample negatives for BPR loss.
    """
    
    def __init__(self, user_sequences, max_seq_len=50, mode='train'):
        self.max_seq_len = max_seq_len
        self.mode = mode
        self.all_items = set()
        
        self.input_seqs = []
        self.target_seqs = []
        self.neg_seqs = []
        self.user_items = []
        self.seq_lens = []
        
        self._build(user_sequences)
    
    def _build(self, user_sequences):
        # Collect all items
        for uid, seq in user_sequences.items():
            self.all_items.update(seq['item_ids'])
        self.all_items_list = list(self.all_items)
        
        for uid, seq in user_sequences.items():
            items = seq['item_ids']
            if len(items) < 3:
                continue
            
            user_item_set = set(items)
            
            if self.mode == 'train':
                # Input: all but last, Target: shifted by 1
                input_items = items[:-2]  # leave last 2 for val/test
                target_items = items[1:-1]
            else:
                input_items = items[:-1]
                target_items = items[1:]
            
            if len(input_items) == 0:
                continue
            
            # Truncate
            if len(input_items) > self.max_seq_len:
                input_items = input_items[-self.max_seq_len:]
                target_items = target_items[-self.max_seq_len:]
            
            # Pad from RIGHT (items first, then zeros)
            # Right-padding is critical: with left-padding + causal mask,
            # padded positions at the start have ALL keys masked -> softmax NaN.
            # With right-padding, position 0 always has a valid item.
            seq_len = len(input_items)
            pad_len = self.max_seq_len - seq_len
            input_padded = input_items + [0] * pad_len
            target_padded = target_items + [0] * pad_len
            
            # Sample negatives for each position
            neg_items = []
            for _ in range(len(input_padded)):
                neg = np.random.choice(self.all_items_list)
                while neg in user_item_set:
                    neg = np.random.choice(self.all_items_list)
                neg_items.append(neg)
            
            self.input_seqs.append(input_padded)
            self.target_seqs.append(target_padded)
            self.neg_seqs.append(neg_items)
            self.user_items.append(user_item_set)
            self.seq_lens.append(seq_len)
    
    def __len__(self):
        return len(self.input_seqs)
    
    def __getitem__(self, idx):
        return {
            'input_ids': torch.LongTensor(self.input_seqs[idx]),
            'target_ids': torch.LongTensor(self.target_seqs[idx]),
            'neg_ids': torch.LongTensor(self.neg_seqs[idx]),
            'seq_len': self.seq_lens[idx]
        }


print('Building SASRec datasets...')
sasrec_train = SASRecDataset(user_sequences, max_seq_len=MAX_SEQ_LEN, mode='train')
sasrec_test = SASRecDataset(user_sequences, max_seq_len=MAX_SEQ_LEN, mode='test')
print(f'SASRec Train: {len(sasrec_train):,}, Test: {len(sasrec_test):,}')

sasrec_train_loader = DataLoader(sasrec_train, batch_size=BATCH_SIZE, shuffle=True,
                                 num_workers=0, pin_memory=True)
sasrec_test_loader = DataLoader(sasrec_test, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=0, pin_memory=True)

## 7. Train SASRec

> **Concept:** SASRec is trained with **BPR (Bayesian Personalized Ranking) loss**, which learns to rank the positive (next) item higher than a sampled negative item:
>
> $$\mathcal{L}_{BPR} = -\sum_{(s, i^+, i^-)} \log \sigma(\hat{r}_{si^+} - \hat{r}_{si^-})$$
>
> This pairwise loss is well-suited for ranking tasks and avoids the need for a calibrated probability output.

In [ ]:
def train_sasrec_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    n_batches = 0
    
    for batch in loader:
        input_ids = batch['input_ids'].to(device)    # (B, S)
        target_ids = batch['target_ids'].to(device)  # (B, S)
        neg_ids = batch['neg_ids'].to(device)        # (B, S)
        
        optimizer.zero_grad()
        
        # Get sequence representations for all positions
        seq_output = model(input_ids, return_all_positions=True)  # (B, S, D)
        
        # Positive and negative scores
        pos_emb = model.item_embedding(target_ids)  # (B, S, D)
        neg_emb = model.item_embedding(neg_ids)     # (B, S, D)
        
        pos_scores = (seq_output * pos_emb).sum(dim=-1)  # (B, S)
        neg_scores = (seq_output * neg_emb).sum(dim=-1)  # (B, S)
        
        # BPR loss (only on non-padded positions)
        # Use torch.where to avoid NaN * 0 = NaN (IEEE 754)
        mask = (target_ids != 0)  # (B, S)
        bpr_loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8)
        loss = torch.where(mask, bpr_loss, torch.zeros_like(bpr_loss))
        loss = loss.sum() / mask.float().sum().clamp(min=1)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / n_batches


def evaluate_sasrec(model, loader, device, k_list=[5, 10, 20]):
    """Evaluate SASRec with HR@K and NDCG@K."""
    model.eval()
    
    metrics = {f'HR@{k}': [] for k in k_list}
    metrics.update({f'NDCG@{k}': [] for k in k_list})
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)    # (B, S)
            target_ids = batch['target_ids'].to(device)  # (B, S)
            neg_ids = batch['neg_ids'].to(device)        # (B, S)
            
            seq_output = model(input_ids, return_all_positions=True)  # (B, S, D)
            
            # Evaluate on the LAST valid position only (right-padded: seq_len - 1)
            B = input_ids.size(0)
            seq_lens = batch["seq_len"].to(device)  # (B,)
            last_pos = seq_lens - 1  # (B,)
            last_pos = last_pos.clamp(min=0)
            
            # Get representation at last valid position
            last_output = seq_output[torch.arange(B, device=device), last_pos]  # (B, D)
            
            # Target at last valid position
            last_target = target_ids[torch.arange(B, device=device), last_pos]  # (B,)
            
            # Score against all items (or sample 100 negatives for efficiency)
            n_neg_eval = 99
            neg_items = torch.randint(1, model.n_items, (B, n_neg_eval), device=device)
            
            # Candidates: [target, neg1, neg2, ...]
            candidates = torch.cat([last_target.unsqueeze(1), neg_items], dim=1)  # (B, 100)
            
            # Scores
            scores = model.predict(last_output, candidates)  # (B, 100)
            
            # Rank: position of the true item (index 0)
            _, indices = scores.sort(dim=1, descending=True)
            ranks = (indices == 0).nonzero(as_tuple=True)[1] + 1  # 1-indexed rank
            
            for k in k_list:
                hr = (ranks <= k).float().cpu().numpy()
                ndcg = (1.0 / torch.log2(ranks.float() + 1) * (ranks <= k).float()).cpu().numpy()
                metrics[f'HR@{k}'].extend(hr.tolist())
                metrics[f'NDCG@{k}'].extend(ndcg.tolist())
    
    return {k: np.mean(v) for k, v in metrics.items()}

In [ ]:
# Train SASRec
sasrec_optimizer = torch.optim.Adam(sasrec_model.parameters(), lr=1e-3, weight_decay=1e-6)
sasrec_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    sasrec_optimizer, mode='max', patience=1, factor=0.5
)

sasrec_best_hr10 = 0
patience_counter = 0
sasrec_history = {'train_loss': [], 'HR@5': [], 'HR@10': [], 'HR@20': [],
                  'NDCG@5': [], 'NDCG@10': [], 'NDCG@20': []}

print(f'Training SASRec for up to {MAX_EPOCHS} epochs...')
print('='*80)

for epoch in range(MAX_EPOCHS):
    start_time = time.time()
    
    train_loss = train_sasrec_epoch(sasrec_model, sasrec_train_loader, sasrec_optimizer, device)
    metrics = evaluate_sasrec(sasrec_model, sasrec_test_loader, device)
    
    epoch_time = time.time() - start_time
    sasrec_scheduler.step(metrics['HR@10'])
    
    sasrec_history['train_loss'].append(train_loss)
    for k in ['HR@5', 'HR@10', 'HR@20', 'NDCG@5', 'NDCG@10', 'NDCG@20']:
        sasrec_history[k].append(metrics[k])
    
    improved = ''
    if metrics['HR@10'] > sasrec_best_hr10:
        sasrec_best_hr10 = metrics['HR@10']
        patience_counter = 0
        torch.save(sasrec_model.state_dict(), os.path.join(PROCESSED_DIR, 'sasrec_best.pt'))
        improved = ' *'
    else:
        patience_counter += 1
    
    print(f'Epoch {epoch+1:2d}/{MAX_EPOCHS} | '
          f'Loss: {train_loss:.4f} | '
          f'HR@5: {metrics["HR@5"]:.4f} | '
          f'HR@10: {metrics["HR@10"]:.4f} | '
          f'HR@20: {metrics["HR@20"]:.4f} | '
          f'NDCG@10: {metrics["NDCG@10"]:.4f} | '
          f'Time: {epoch_time:.1f}s{improved}')
    
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break

print(f'\nSASRec Best HR@10: {sasrec_best_hr10:.4f}')

In [ ]:
# Plot SASRec training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs = range(1, len(sasrec_history['train_loss']) + 1)

axes[0].plot(epochs, sasrec_history['train_loss'], 'o-', color='#4e79a7', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BPR Loss')
axes[0].set_title('SASRec: Training Loss')

for k, color in zip(['HR@5', 'HR@10', 'HR@20'], ['#4e79a7', '#59a14f', '#f28e2b']):
    axes[1].plot(epochs, sasrec_history[k], 'o-', label=k, color=color, linewidth=2)
axes[1].axhline(0.50, color='red', linestyle='--', alpha=0.7, label='Target HR@10=0.50')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Hit Rate')
axes[1].set_title('SASRec: Hit Rate@K')
axes[1].legend()

for k, color in zip(['NDCG@5', 'NDCG@10', 'NDCG@20'], ['#4e79a7', '#59a14f', '#f28e2b']):
    axes[2].plot(epochs, sasrec_history[k], 'o-', label=k, color=color, linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('NDCG')
axes[2].set_title('SASRec: NDCG@K')
axes[2].legend()

plt.tight_layout()
plt.savefig('sasrec_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Multi-Head Attention Visualization

> **Concept:** Visualizing attention patterns across multiple heads reveals that different heads learn to capture different types of relationships: some focus on recent items (recency bias), while others capture category-level patterns.

In [ ]:
# Extract attention weights from BST
# We need to register hooks to capture attention weights
bst_model.load_state_dict(torch.load(os.path.join(PROCESSED_DIR, 'bst_best.pt'), map_location=device))
bst_model.eval()

attention_weights = []

def hook_fn(module, input, output):
    # output is (attn_output, attn_weights) for MultiheadAttention
    if isinstance(output, tuple) and len(output) == 2:
        attention_weights.append(output[1].detach().cpu())

# Register hooks on self-attention layers
hooks = []
for layer in bst_model.transformer_encoder.layers:
    hook = layer.self_attn.register_forward_hook(hook_fn)
    hooks.append(hook)

# Need to set need_weights=True
# Temporarily modify forward to get attention weights
for layer in bst_model.transformer_encoder.layers:
    layer.self_attn.need_weights = True
    layer.self_attn.average_attn_weights = False  # get per-head weights

# Forward pass
sample_batch = next(iter(test_loader))
with torch.no_grad():
    hist_items = sample_batch['hist_items'].to(device)
    hist_cats = sample_batch['hist_cats'].to(device)
    hist_len = sample_batch['hist_len'].to(device)
    target_item = sample_batch['target_item'].to(device)
    target_cat = sample_batch['target_cat'].to(device)
    _ = bst_model(hist_items, hist_cats, hist_len, target_item, target_cat)

# Clean up hooks
for h in hooks:
    h.remove()

print(f'Captured {len(attention_weights)} attention weight tensors')
if attention_weights:
    print(f'Shape: {attention_weights[0].shape}')  # (B, n_heads, S, S)

In [ ]:
# Visualize multi-head attention for a sample user
if attention_weights:
    sample_idx = 0
    # Find a sample with reasonable sequence length
    for i in range(len(hist_items)):
        valid_len = (sample_batch['hist_items'][i] != 0).sum().item()
        if 15 <= valid_len <= 25:
            sample_idx = i
            break
    
    valid_mask = sample_batch['hist_items'][sample_idx] != 0
    valid_len = valid_mask.sum().item()
    
    # Layer 1 attention
    layer1_attn = attention_weights[0][sample_idx].numpy()  # (n_heads, S, S)
    n_heads = layer1_attn.shape[0]
    
    # Only show valid positions
    start_pos = MAX_SEQ_LEN - valid_len
    
    fig, axes = plt.subplots(1, min(n_heads, 4), figsize=(16, 4))
    if n_heads == 1:
        axes = [axes]
    
    for h in range(min(n_heads, 4)):
        attn_map = layer1_attn[h, start_pos:, start_pos:]
        im = axes[h].imshow(attn_map, cmap='Blues', aspect='auto')
        axes[h].set_title(f'Head {h+1}')
        axes[h].set_xlabel('Key Position')
        if h == 0:
            axes[h].set_ylabel('Query Position')
        plt.colorbar(im, ax=axes[h], fraction=0.046)
    
    plt.suptitle(f'BST Layer 1: Multi-Head Self-Attention (seq_len={valid_len})',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('bst_attention_heads.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No attention weights captured. This may be due to PyTorch version differences.')
    print('The hook approach requires PyTorch to return attention weights from MultiheadAttention.')

## 9. Ablation Studies

> **Pro Tip:** When tuning Transformer hyperparameters for sequential recommendation, the most impactful factors are typically: (1) number of layers, (2) embedding dimension, and (3) number of attention heads. More is not always better due to overfitting on sparse interaction data.

In [ ]:
# BST Ablation: number of heads and layers
ablation_configs = [
    {'n_heads': 1, 'n_layers': 1, 'name': '1H-1L'},
    {'n_heads': 4, 'n_layers': 1, 'name': '4H-1L'},
    {'n_heads': 4, 'n_layers': 2, 'name': '4H-2L'},
]

ablation_results = {}

for config in ablation_configs:
    print(f"\nTraining BST ({config['name']})...")
    
    abl_model = BST(
        n_items=n_items, n_categories=n_categories,
        embed_dim=32, n_heads=config['n_heads'], n_layers=config['n_layers'],
        max_seq_len=MAX_SEQ_LEN, hidden_dims=[256, 128, 64], dropout=0.2
    ).to(device)
    
    abl_optimizer = torch.optim.Adam(abl_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    abl_best_auc = 0
    patience_cnt = 0
    
    for epoch in range(MAX_EPOCHS):
        train_loss = train_epoch(abl_model, train_loader, abl_optimizer, criterion, device)
        _, test_auc, _, _, _ = evaluate(abl_model, test_loader, criterion, device)
        
        if test_auc > abl_best_auc:
            abl_best_auc = test_auc
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= PATIENCE:
            break
    
    ablation_results[config['name']] = abl_best_auc
    n_params = sum(p.numel() for p in abl_model.parameters())
    print(f"  {config['name']}: AUC = {abl_best_auc:.4f}, Params = {n_params:,}")

print('\n' + '='*50)
print('BST Ablation Results:')
for name, auc in sorted(ablation_results.items(), key=lambda x: x[1], reverse=True):
    print(f'  {name:10s}: AUC = {auc:.4f}')

In [ ]:
# Plot ablation results
fig, ax = plt.subplots(figsize=(10, 5))

names = list(ablation_results.keys())
aucs = [ablation_results[n] for n in names]

bars = ax.bar(names, aucs, color='#4e79a7', edgecolor='white', linewidth=2)
ax.axhline(0.64, color='red', linestyle='--', alpha=0.7, label='Target AUC=0.64')
ax.set_ylabel('Test AUC')
ax.set_title('BST Architecture Ablation (Heads x Layers)')
ax.legend()

for bar, auc_val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
            f'{auc_val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

min_auc = min(aucs)
ax.set_ylim(min_auc - 0.02, max(aucs) + 0.02)

plt.tight_layout()
plt.savefig('bst_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Exercises

### Exercise 1: Sinusoidal Positional Encoding
Replace the learnable positional embeddings in BST with fixed sinusoidal positional encodings (as in the original Transformer paper). Compare performance.

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

```python
# TODO: Your code here
```

### Exercise 2: Target-Aware BST
Modify BST to include the target item in the Transformer input (appended to the sequence), then use only the target position's output for prediction. Compare with the current mean-pooling approach.

```python
# TODO: Your code here
# Hint: Append target_emb to the sequence before feeding to the Transformer
```

### Exercise 3: SASRec with Cross-Entropy Loss
Replace BPR loss with cross-entropy (softmax) loss over a sampled set of negatives. Compare convergence speed and final performance.

```python
# TODO: Your code here
# Hint: Use nn.CrossEntropyLoss with the positive item as the target class
```

### Exercise 4: Relative Position Encoding
Implement relative positional encoding (as in Transformer-XL or ALiBi) for SASRec. Instead of adding absolute position embeddings, bias the attention scores based on the distance between positions.

```python
# TODO: Your code here
```

---

## Summary and Key Takeaways

1. **BST vs DIN**: BST captures pairwise item-item relationships within the sequence through self-attention, going beyond DIN's target-only attention. This leads to improved AUC for CTR prediction.

2. **SASRec's Causal Attention**: By using left-to-right masking, SASRec learns a generative model of item sequences, predicting the next item at each position. This autoregressive formulation naturally captures sequential dependencies.

3. **Padding Masking is Critical**: Without proper padding masks in the Transformer, padded positions contaminate the attention computation. This is the most common implementation bug in sequential recommendation Transformers.

4. **Architecture Choices Matter**: The ablation study reveals that increasing model capacity (more heads, more layers) helps up to a point, after which overfitting on sparse data dominates.

5. **Different Heads, Different Patterns**: Multi-head attention visualization shows that different heads specialize in different patterns (recency, category similarity, long-range dependencies).

6. **Task Formulation**: BST (CTR with AUC) and SASRec (next-item with HR@K) solve related but different tasks. The choice depends on the application — CTR for ad ranking, next-item for recommendation feeds.

### Next Steps
In the final notebook, we perform a comprehensive **head-to-head comparison** of all models (Mean Pooling, DIN, BST, SASRec), analyzing their strengths and weaknesses across multiple dimensions.